In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from datetime import date, datetime, timedelta
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import fcluster
from matplotlib.colors import TwoSlopeNorm

# parameter

In [ ]:
yeari, yearf = '2024', '2024'
weeki, weekf = '18', '31'

In [ ]:
di = datetime.strptime(f'{yeari}-{weeki}-1', "%Y-%W-%w").date()
df = datetime.strptime(f'{yearf}-{weekf}-1', "%Y-%W-%w").date() + timedelta(6)
ds = [di+timedelta(dt) for dt in range((df-di).days+1)]
daylist = ds
print(di, 'until', df)

In [ ]:
cdef = 'tl7_10m'# 'tl5_10m' 'tl6_10m' 'tl7_10m' 'tl8_10m' 'tl8_60m'
cdef_alt = '16m_10min'# tl5: 62 ... tl7: 16   tl8: 8

# figures & tables

In [ ]:
data_osm2 = pd.read_csv('data/fig5/contacts_matchdays_osmmapped.csv', low_memory=False)
data_osmpoint2 = pd.read_csv('data/fig5/contacts_matchdays_osmpointmapped2.csv', low_memory=False)
data_hw2 = pd.read_csv('data/fig5/contacts_matchdays_homework.csv', low_memory=False).rename(columns={'is_home':'home','is_work':'work'})
data_hw2['home'] = data_hw2.home.map({True:'yes',False:'no'})
data_hw2['work'] = data_hw2.work.map({True:'yes',False:'no'})
print('data_osm2', len(data_osm2), len(data_osm2.columns))
print('data_osmpoint2', len(data_osmpoint2), len(data_osmpoint2.columns))
print('data_hw2', len(data_hw2), len(data_hw2.columns))

In [ ]:
data_osm2 = data_osm2[data_osm2.city.isin(['Stuttgart','Frankfurt am Main','Dortmund','München'])]
data_osmpoint2 = data_osmpoint2[data_osmpoint2.city.isin(['Stuttgart','Frankfurt am Main','Dortmund','München'])]
data_hw2 = data_hw2[data_hw2.city.isin(['Stuttgart','Frankfurt am Main','Dortmund','München'])]
print('data_osm2', len(data_osm2), len(data_osm2.columns))
print('data_osmpoint2', len(data_osmpoint2), len(data_osmpoint2.columns))
print('data_hw2', len(data_hw2), len(data_hw2.columns))

In [ ]:
data_osm2_bef = data_osm2[(data_osm2.hour_rel.isin([-2,-1]))].copy() # [(data_osm2.hour_rel < 0)].copy() [(data_osm2.hour_rel.isin([-2,-1]))].copy()
data_osm2_bef['match_rel'] = 'before match'
data_osm2_dur = data_osm2[(data_osm2.hour_rel.isin([0,1]))].copy()
data_osm2_dur['match_rel'] = 'during match'
data_osm2_aft = data_osm2[(data_osm2.hour_rel.isin([2,3]))].copy() # [(data_osm2.hour_rel > 1)].copy() [(data_osm2.hour_rel.isin([2,3]))].copy()
data_osm2_aft['match_rel'] = 'after match'
data_osm2_all = data_osm2.copy()
data_osm2_all['match_rel'] = 'day of the match'
data_osm2 = pd.concat([data_osm2_bef, data_osm2_dur, data_osm2_aft])#, data_osm2_all])

data_osmpoint2_bef = data_osmpoint2[(data_osmpoint2.hour_rel.isin([-2,-1]))].copy() # [(data_osmpoint2.hour_rel < 0)].copy() [(data_osmpoint2.hour_rel.isin([-2,-1]))].copy()
data_osmpoint2_bef['match_rel'] = 'before match'
data_osmpoint2_dur = data_osmpoint2[(data_osmpoint2.hour_rel.isin([0,1]))].copy()
data_osmpoint2_dur['match_rel'] = 'during match'
data_osmpoint2_aft = data_osmpoint2[(data_osmpoint2.hour_rel.isin([2,3]))].copy() # [(data_osmpoint2.hour_rel > 1)].copy() [(data_osmpoint2.hour_rel.isin([2,3]))].copy()
data_osmpoint2_aft['match_rel'] = 'after match'
data_osmpoint2_all = data_osmpoint2.copy()
data_osmpoint2_all['match_rel'] = 'day of the match'
data_osmpoint2 = pd.concat([data_osmpoint2_bef, data_osmpoint2_dur, data_osmpoint2_aft])#, data_osmpoint2_all])

data_hw2_bef = data_hw2[(data_hw2.hour_rel.isin([-2,-1]))].copy()
data_hw2_bef['match_rel'] = 'before match'
data_hw2_dur = data_hw2[(data_hw2.hour_rel.isin([0,1]))].copy()
data_hw2_dur['match_rel'] = 'during match'
data_hw2_aft = data_hw2[(data_hw2.hour_rel.isin([2,3]))].copy()
data_hw2_aft['match_rel'] = 'after match'
data_hw2_all = data_hw2.copy()
data_hw2_all['match_rel'] = 'day of the match'
data_hw2 = pd.concat([data_hw2_bef, data_hw2_dur, data_hw2_aft])#, data_hw2_all])

print(len(data_osm2))
print(len(data_osmpoint2))
print(len(data_hw2))

In [ ]:
df_norm_bytime = data_osm2.groupby(['germany','match_rel']).apply(len).reset_index().rename(columns={0:'norm'})
df_norm_bytime

In [ ]:
df_norm = data_osm2.groupby(['germany']).apply(len).reset_index().rename(columns={0:'norm'})
df_norm

In [ ]:
attr_freq = pd.DataFrame()
excl = ['germany','match_rel','city','hour_rel','osm_id','osm_id.1','plz','tile_id','tile_id.1','tl','z_order','osm_id_point']
for attr in [col for col in data_osm2.columns if col not in excl]:
    if (~data_osm2[attr].isna()).sum() > 0:
        df_cnt = data_osm2.groupby(['germany','match_rel'])[attr].apply(lambda x: (~x.isna()).sum()).reset_index()\
                    .merge(df_norm_bytime, on=['germany','match_rel'])
        df_cnt['freq'] = df_cnt[attr] / df_cnt.norm
        df_cnt['attr'] = attr
        attr_freq = pd.concat([attr_freq, df_cnt[['germany','match_rel','attr','freq']]])
    else:
        print(f'no contacts: {attr}')
for attr in [col for col in data_osmpoint2.columns if col not in excl]:
    if (~data_osmpoint2[attr].isna()).sum() > 0:
        df_cnt = data_osmpoint2.groupby(['germany','match_rel'])[attr].apply(lambda x: (~x.isna()).sum()).reset_index()\
                    .merge(df_norm_bytime, on=['germany','match_rel'])
        df_cnt['freq'] = df_cnt[attr] / df_cnt.norm
        df_cnt['attr'] = attr
        attr_freq = pd.concat([attr_freq, df_cnt[['germany','match_rel','attr','freq']]])
    else:
        print(f'no contacts: {attr}')
for attr in [col for col in data_hw2.columns if col not in excl]:
    if (~data_hw2[attr].isna()).sum() > 0:
        df_cnt = data_hw2.groupby(['germany','match_rel'])[attr].apply(lambda x: (~x.isna()).sum()).reset_index()\
                    .merge(df_norm_bytime, on=['germany','match_rel'])
        df_cnt['freq'] = df_cnt[attr] / df_cnt.norm
        df_cnt['attr'] = attr
        attr_freq = pd.concat([attr_freq, df_cnt[['germany','match_rel','attr','freq']]])
    else:
        print(f'no contacts: {attr}')
attr_freq['germany'] = attr_freq.germany.astype(str)
attr_freq#.groupby(['germany','match_rel']).freq.apply(np.sum).reset_index()

In [ ]:
attr_freq_toplot = attr_freq.copy(deep=True)
attr_freq_toplot['attr'] = [at if '_point' not in at else '*'+at[:-6] for at in attr_freq_toplot.attr]
attr_freq_toplot

In [ ]:
sns.set_theme(style="ticks")

# Create a FacetGrid, grouping by 'category'
g = sns.FacetGrid(attr_freq_toplot.sort_values(['attr','germany']),
                  row='match_rel', margin_titles=True, height=2, aspect=7.5,
                  row_order=['before match','during match','after match'],
                  legend_out=True,)#,'day of the match'])

# Map the custom heatmap function to each facet
g.map_dataframe(sns.barplot, x='attr', y='freq', hue='germany', palette=sns.color_palette(None, 2))

# Set yticks for all subplots
g.set(xlabel='OSM key', ylabel='contact share')#, yticks=[])
#g.set_yticklabels([], rotation=0)
#g.set_xticklabels([], rotation=0)

for ax in g.axes.flat:
    ax.set_yscale('symlog', linthresh=.01)

# Modify margin titles
g.set_titles(row_template="{row_name}")#, size=14, fontweight='bold')
g.axes.flat[-1].set_xticklabels(g.axes.flat[-1].get_xticklabels(), rotation=90)

# legend
g.add_legend()
lg = g._legend
lg.set_title('German match')
for tx in lg.texts:
    if tx.get_text() == 'False':
        tx.set_text('no')
    elif tx.get_text() == 'True':
        tx.set_text('yes')

g.tight_layout()
plt.show()

In [ ]:
attr2vals = {
    'aeroway':['terminal'],
    'amenity':['parking','biergarten','full','nightclub','police','pub','restaurant','shelter','marketplace'],
    'area':['yes'],
    'barrier':['fence','bollard','wall'],
    'bicycle':['yes'],
    'building':['yes','train_station','stadium','roof','retail','grandstand'],
    'foot':['yes','no','permissive','designated'],
    'highway':['primary','secondary','tertiary','pedestrian','motorway','footway','cycleway','service'],
    'landuse':['residential','retail','railway','commercial'],
    'leisure':['stadium','pitch','park'],
    'oneway':['yes'],
    'railway':['light_rail','platform','rail','subway','tram'],
    'route':['tram','train','subway','road','light_rail','bus','bicycle'],
    'service':['tourism','event','driveway','commuter'],
    'shop':['supermarket','mall'],
    'sport':['soccer'],
    'surface':['paving stones','paved','grass','fine_gravel','asphalt','sett'],
    'tourism':['attraction','museum'],
}
attr2vals_point = {
    'amenity_point':['atm','bank','bar','bench','bicycle_rental','cafe','community_centre','drinking_water','fast_food',
                     'locker','lounge','parking_entrance','police','pub','restaurant','taxi','telephone','toilets',
                     'vending_machine','waste_basket'],
    'shop_point':[v for v in set(data_osmpoint2[~data_osmpoint2.shop_point.isna()].shop_point)],
    'shop_bool_point':['yes'],
    'tourism_point':['artwork','hotel','information','museum','viewpoint'],
}
attr2vals_hw = {
    'home':['yes'],
    'work':['yes'],
}

In [ ]:
cat_freq = pd.DataFrame()
data_osmpoint2['shop_bool_point'] = (~data_osmpoint2.shop_point.isna()).astype(str).map({'True':'yes','False':'no'})
for attr, vals in attr2vals.items():
    df_cnt = data_osm2.groupby(['germany','match_rel',attr]).apply(lambda x: len(x)).reset_index()\
                    .rename(columns={0:'cnt', attr:'cat'}).merge(df_norm, on=['germany'])#'match_rel','germany'
    df_cnt = df_cnt[df_cnt.cat.isin(vals)]
    df_cnt['cat'] = [attr+':'+c for c in df_cnt.cat]
    cat_freq = pd.concat([cat_freq, df_cnt])
for attr, vals in attr2vals_point.items():
    df_cnt = data_osmpoint2.groupby(['germany','match_rel',attr]).apply(lambda x: len(x)).reset_index()\
                    .rename(columns={0:'cnt', attr:'cat'}).merge(df_norm, on=['germany'])#'match_rel','germany'
    df_cnt = df_cnt[df_cnt.cat.isin(vals)]
    df_cnt['cat'] = [attr+':'+c for c in df_cnt.cat]
    cat_freq = pd.concat([cat_freq, df_cnt])
for attr, vals in attr2vals_hw.items():
    df_cnt = data_hw2.groupby(['germany','match_rel',attr]).apply(lambda x: len(x)).reset_index()\
                    .rename(columns={0:'cnt', attr:'cat'}).merge(df_norm, on=['germany'])#'match_rel','germany'
    df_cnt = df_cnt[df_cnt.cat.isin(vals)]
    df_cnt['cat'] = [attr+':'+c for c in df_cnt.cat]
    cat_freq = pd.concat([cat_freq, df_cnt])
cat_freq['freq'] = cat_freq.cnt / cat_freq.norm
cat_freq['germany'] = cat_freq.germany.astype(str)
#cat_freq['cl'] = cat_freq.cat.map(attr2cl)
cat_freq['cat_label'] = cat_freq.cat.apply(lambda x: ':'.join([word if '_point' not in word else '*'+word[:-6] for word in x.split(':')]))
cat_freq

In [ ]:
cat_freq.groupby(['germany']).cnt.sum()

In [ ]:
len(set(cat_freq[cat_freq.match_rel!='day of the match'].cat_label))

In [ ]:
cats_pop = cat_freq.groupby('cat_label').freq.max().reset_index().sort_values('freq', ascending=False)
for _, row in cats_pop.iterrows():
    print(f'{row.cat_label.ljust(30)} {row.freq}')

In [ ]:
# take only map features above minimum contact frequency
cat_freq = cat_freq[cat_freq.cat_label.isin(cats_pop[cats_pop.freq>=5e-3].cat_label.tolist())]

In [ ]:
# gather data for clustering of map features
to_cluster = cat_freq[(cat_freq.match_rel!='day of the match') & (cat_freq.cat_label!='*shop_bool:yes')][['germany','match_rel','cat_label','freq']]
#'cnt','norm',
# if necessary: remove map features with too few contacts
to_cluster15 = to_cluster.groupby('cat_label').freq.apply(np.sum).reset_index()
cats_restr = to_cluster15[to_cluster15.freq >= 0*to_cluster15.freq.min()].sort_values('freq', ascending=False).cat_label.tolist()# 10*
#cats_restr = [cat for cat in cats_restr if not cat.startswith('*shop:') or cat=='*shop:yes']
print(f'leftover map features: {len(cats_restr)}')
# get data vector for each map feature
to_cluster2 = to_cluster[to_cluster.cat_label.isin(cats_restr)].pivot(index=['germany','match_rel'], columns='cat_label', values='freq').fillna(0)
# normalize the vectors
scaler = StandardScaler()
normalized = scaler.fit_transform(to_cluster2)
to_cluster3 = pd.DataFrame(normalized, index=to_cluster2.index, columns=to_cluster2.columns)
to_cluster3

In [ ]:
cats = [s.split(':')[0] for s in to_cluster3.columns.tolist()]
clist = sns.husl_palette(len(set(cats)), s=1.)
cat2col = dict(zip(list(set(cats)), clist))
row_cols = pd.DataFrame(cats, index=to_cluster3.columns, columns=['cat_label']).cat_label.map(cat2col)

interesting clustering with deep hierarchies obtained with:

- average/complete, braycurtis
- complete, correlation/cosine/cityblock
- ward, euclidean

In [ ]:
# Draw the full plot
g = sns.clustermap(to_cluster3.corr(), center=0, cmap="vlag",
                   method='ward',# 'average','single','complete','weighted','centroid','median','ward'
                   metric='euclidean',# 'euclidean','cityblock','cosine','correlation','hamming','jaccard','chebyshev','canberra','braycurtis','matching'
                   row_colors=row_cols, col_colors=row_cols,
                   dendrogram_ratio=(.1, .2),
                   cbar_pos=(.02, .32, .03, .2),
                   linewidths=0., figsize=(12, 13))

g.ax_row_dendrogram.remove()

In [ ]:
# get clusters at desired dendrogram depth
depth = 3#11# no. of clusters
linkagemat = g.dendrogram_row.linkage
cluster_labels = fcluster(linkagemat, t=depth, criterion='maxclust')# Map the cluster labels to the original row labels
row_labels = pd.DataFrame(pd.Series(cluster_labels, index=to_cluster3.corr().index), columns=['cluster'])
row_cats = pd.DataFrame(row_cols).rename(columns={'day':'category'})
cats_clus = row_labels.join(row_cats).rename(columns={'cat_label':'color'}).reset_index()
cats_clus

In [ ]:
# plot the contact trends in each cluster
cats_h = cats_clus.cat_label#[cats_clus.cluster==1]
to_cluster4 = to_cluster3[cats_h].stack('cat_label').reset_index().rename(columns={0:'freq'})#.groupby(['germany','match_rel']).freq.apply(np.mean)
to_cluster4['trel'] = to_cluster4.match_rel.map({'before match':0, 'during match':1, 'after match':2})
to_cluster4 = to_cluster4.merge(cats_clus[['cat_label','cluster']], on='cat_label')
to_cluster4

In [ ]:
# Define the palette as a list to specify exact values
palette = sns.color_palette("rocket_r")

# Plot the lines on two facets
sns.relplot(
    data=to_cluster4[['germany','match_rel','freq','trel','cat_label','cluster']],
    x="trel", y="freq",
    col="germany", row="cluster", hue="cat_label",# size="cat_label",
    kind="line", size_order=["T1", "T2"], palette=palette,
    height=3, aspect=1., facet_kws=dict(sharex=False), legend=False,
)

In [ ]:
for _, row in to_cluster4.groupby('cluster').cat_label.apply(list).reset_index().iterrows():
    print(row.cluster)
    print(' '.join(sorted(list(set(row.cat_label)))))
    print()

In [ ]:
# colors for colorbar for clusters
#cluster_soll = 12
#cats = [cats_clus[cats_clus.cat_label==s].cluster.iloc[0]==cluster_soll for s in to_cluster3.columns.tolist()]
cats = [cats_clus[cats_clus.cat_label==s].cluster.iloc[0] for s in to_cluster3.columns.tolist()]
clist = sns.husl_palette(len(set(cats)), s=1.)
cat2col = dict(zip(list(set(cats)), clist))
row_cols2 = pd.DataFrame(cats, index=to_cluster3.columns, columns=['cat_label']).cat_label.map(cat2col)

In [ ]:
sns.set_theme(style="ticks")
# Draw the full plot
cnorm = TwoSlopeNorm(vmin=-1., vcenter=0., vmax=1.)
g = sns.clustermap(to_cluster3.corr(), center=0, cmap="vlag_r", norm=cnorm,
                   method='ward',# 'average','single','complete','weighted','centroid','median','ward'
                   metric='euclidean',# 'euclidean','cityblock','cosine','correlation','hamming','jaccard','chebyshev','canberra','braycurtis','matching'
                   row_colors=row_cols2, col_colors=row_cols2,
                   dendrogram_ratio=(.1, .2),
                   cbar_pos=(.8, .445, .02, .2),#(.775, .425, .02, .2),#cbar_pos=(.02, .32, .03, .2),
                   linewidths=0., figsize=(7,7.55),# square=True, 
                  )

#g.ax_row_dendrogram.remove()
g.ax_col_dendrogram.remove()
g.ax_heatmap.set_xticks([])
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_xlabel('OSM map feature', fontsize=15)
g.ax_heatmap.set_ylabel('OSM map feature', fontsize=15)
g.ax_row_colors.set_xticks([])
g.ax_col_colors.set_yticks([])
g.cax.set_ylim([-1,1])
g.cax.set_yticks([-1,-.5,0,.5,1])
g.cax.tick_params(labelsize=15)

plt.savefig(f'plots/fig4/contacts_around_euro24_matches_poi_clustering.jpg', bbox_inches='tight', dpi=300)
plt.savefig(f'plots/fig4/contacts_around_euro24_matches_poi_clustering.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# define labels for the clusters and join them onto the data
#clcm2label = {
#    1:'routine & commute', 2:'travel', 3:'event venue', 4:'underground', 5:'pre-event off-venue', 6:'off-venue (1)', 7:'off-venue (2)',
#    8:'suppressed (1)', 9:'suppressed (2)', 10:'home', 11:'post-event off-venue'
#}
clcm2label = {
    #nn: f'cluster {nn}' for nn in range(1,1+depth)
    1:'enriched contact settings', 2:'depleted contact settings', 3:'redistributed contact settings'
}
cat2clcm = to_cluster4.groupby('cluster').cat_label.apply(lambda x: list(set(x))).reset_index().explode('cat_label').rename(columns={'cluster':'cl_cm'})
cat_freq_cm = cat_freq.merge(cat2clcm, on='cat_label')
cat_freq_cm['cl_cm_label'] = cat_freq_cm.cl_cm.map(clcm2label)
cat_freq_cm

In [ ]:
df_norm_again = cat_freq_cm[(cat_freq_cm.match_rel!='day of the match')].groupby(['germany']).cnt.sum().reset_index().rename(columns={'cnt':'norm'})
cat_freq_cm = cat_freq_cm.drop(columns=['norm']).merge(df_norm_again, on='germany')
cat_freq_cm['freq'] = cat_freq_cm.cnt / cat_freq_cm.norm
cat_freq_cm

In [ ]:
#cls_ordered_cm = [f'cluster {nn}' for nn in range(1,1+depth)]
#cls_ordered_cm = ['routine & commute', 'travel', 'event venue', 'underground', 'pre-event off-venue', 'off-venue (1)', 'off-venue (2)',
#    'suppressed (1)', 'suppressed (2)', 'home', 'post-event off-venue']
cls_ordered_cm = ['enriched contact settings', 'depleted contact settings', 'redistributed contact settings']
cats_ordered_cm = []
xclims_cm = [0]
for cl_cm in cls_ordered_cm:
    cats_ordered_cm += sorted(set(cat_freq_cm[cat_freq_cm.cl_cm_label==cl_cm].cat_label))
    xclims_cm.append(len(cats_ordered_cm))

In [ ]:
sns.set_theme(style="ticks")

# Create a FacetGrid, grouping by 'category'
g = sns.FacetGrid(cat_freq_cm[cat_freq_cm.match_rel!='day of the match'].sort_values(['cl_cm','cat','germany']),
                  row='match_rel', margin_titles=True, height=2.5, aspect=6,#height=2, aspect=5,# height=5, aspect=2.5,
                  row_order=['before match','during match','after match'],#,'day of the match'],
                  #col_order=['commute','attending','auxiliary','ignoring','other']
                 )

# Map the custom heatmap function to each facet
g.map_dataframe(sns.barplot, x='cat_label', y='freq', hue='germany',
                palette=sns.color_palette(None, 2), order=cats_ordered_cm, hue_order=['False','True'],
                lw=0)

# Set yticks for all subplots
g.set(xlabel='OSM map feature', ylabel='contact share')#, yticks=[])
#g.set_yticklabels([], rotation=0)
#g.set_xticklabels([], rotation=0)

# Modify margin titles
g.set_titles(row_template="{row_name}", size=15)#, col_template="{col_name}")#, size=14, fontweight='bold')
g.axes.flat[-1].set_xticklabels(g.axes.flat[-1].get_xticklabels(), rotation=90)

yu = 1.1*cat_freq_cm.freq.max()
linthresh = .01
for j, ax in enumerate(g.axes.flat):
    ax.set_yscale('symlog', linthresh=linthresh)
    ax.plot([-1.,xclims_cm[-1]], [linthresh,linthresh], c='k', ls='--', zorder=-4)
    ax.set_xlim([-1.,xclims_cm[-1]])
    for clcnt, (xl, xr, cl, c_bg) in enumerate(zip(xclims_cm[:-1], xclims_cm[1:], cls_ordered_cm, clist)):
        #alpha = .2 if clcnt%2==0 else 0.# .2
        #ax.fill_between([xl-.5, xr-.5], [yu, yu], color='gray', alpha=alpha, zorder=-5)
        ax.fill_between([xl-.5, xr-.5], [yu, yu], color=c_bg, alpha=.2, zorder=-5)
        if j==0:
            if clcnt%2==0:
                ax.text((xl+xr-1.)/2., yu/2, cl, va='center', ha='center', fontsize=15)
            else:
                ax.text((xl+xr-1.)/2., yu/2, cl, va='center', ha='center', fontsize=15)#yu/1
    ax.tick_params(labelsize=15)
    ax.set_ylabel('contact share', fontsize=15)
g.axes.flat[-1].set_xlabel('OSM map feature', fontsize=15)

#g.add_legend(title='German participation')

g.tight_layout()
plt.savefig(f'plots/figs8/contacts_around_euro24_matches_poi_clustering2.jpg', bbox_inches='tight', dpi=300)
plt.savefig(f'plots/figs8/contacts_around_euro24_matches_poi_clustering2.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# map feature weights
for _, row in to_cluster4.groupby('cluster').cat_label.apply(list).reset_index().iterrows():
    print(row.cluster)
    #cat_labels_here = sorted(list(set(row.cat_label)))
    #print(' '.join(cat_labels_here))
    ws = cat_freq_cm[(cat_freq_cm.match_rel!='day of the match') & (cat_freq_cm.cl_cm==row.cluster)]
    ws = ws.groupby('cat_label').cnt.sum().reset_index().sort_values('cnt', ascending=False)
    for _, row in ws.iterrows():
        print(row.cat_label, row.cnt)
    #print(' '.join(ws.groupby('cat_label').cnt.sum().reset_index().sort_values('cnt', ascending=False).cat_label.tolist()))
    print()

In [ ]:
# order clusters and remove unidentified clusters
cls_ordered_cm_red = [cl for cl in cls_ordered_cm]# if 'cluster ' not in cl and cl not in ['underground','routine & commute (2)','routine & commute (3)','suppressed (1)','suppressed (2)']]
# sum contact frequencies by cluster
cat_freq_cm_agg = cat_freq_cm[cat_freq_cm.cl_cm_label.isin(cls_ordered_cm_red)]\
                        .groupby(['germany','match_rel','cl_cm_label']).freq.apply(np.sum).reset_index()
cat_freq_cm_agg['trel'] = cat_freq_cm_agg.match_rel.map({'before match':0, 'during match':1, 'after match':2})
cat_freq_cm_agg['cl_cm_label'] = pd.Categorical(cat_freq_cm_agg.cl_cm_label, categories=cls_ordered_cm_red, ordered=True)
cat_freq_cm_agg

In [ ]:
# Initialize a grid of plots with an Axes for each walk
ncol = 4
grid = sns.FacetGrid(cat_freq_cm_agg.sort_values(['cl_cm_label','trel']),
                     col="cl_cm_label", hue="germany", palette=sns.color_palette(), col_wrap=ncol,
                     height=2.5, aspect=1.25,
                     sharey=False, legend_out=True)

# Draw a horizontal line to show the starting point
#grid.refline(y=0, linestyle=":")

# Draw a line plot to show the trajectory of each random walk
grid.map(plt.plot, "trel", "freq", marker="o", ms=15, lw=4)

# Adjust the tick positions and labels
grid.set(xticks=range(2+1), xticklabels=['before','during','after'], xlabel='match')#, ylabel='contact share')
#, yticks=[-3, 3], xlim=(-.5, 4.5), ylim=(-3.5, 3.5))
grid.set_axis_labels(x_var='match', fontsize=15)

#linthresh = .01
for j, (ax, l) in enumerate(zip(grid.axes.flat, cls_ordered_cm_red)):
    yup_h = 1.1*cat_freq_cm_agg[cat_freq_cm_agg.cl_cm_label==l].freq.max()
    xl, xr = -.1, 2.1
    ax.set_ylim([0., yup_h])
    ax.set_xlim([xl, xr])
    if j%ncol==0:
        ax.set_ylabel('contact share', fontsize=15)
    #ax.set_yscale('symlog', linthresh=linthresh)
    ax.fill_between([xl,.5], [yup_h,yup_h], color='gray', zorder=0, alpha=.25)
    ax.fill_between([.5,1.5], [yup_h,yup_h], color='gray', zorder=0, alpha=.5)
    ax.fill_between([1.5,xr], [yup_h,yup_h], color='gray', zorder=0, alpha=.25)
    ax.tick_params(labelsize=15)

grid.set_titles(col_template="{col_name}", size=15)

# legend
grid.add_legend()
# Move legend to the right
grid._legend.set_bbox_to_anchor((.7, 0.5))  # (x, y)
grid._legend.set_loc("center left")  # legend aligned to the center-left of the anchor
lg = grid._legend
lg.set_title('German match', prop={'size': 15})
for tx in lg.texts:
    if tx.get_text() == 'False':
        tx.set_text('no')
    elif tx.get_text() == 'True':
        tx.set_text('yes')
    tx.set_fontsize(15)

# Adjust the arrangement of the plots
#grid.fig.tight_layout()#w_pad=1)
plt.savefig(f'plots/fig4/contacts_around_euro24_matches_poi_clustering3.jpg', bbox_inches='tight', dpi=300)
plt.savefig(f'plots/fig4/contacts_around_euro24_matches_poi_clustering3.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# should sum to one, unless some clusters are excluded
cat_freq_cm_agg.groupby(['germany']).freq.sum()

In [ ]:
cat_freq_cm_agg_cmpl = cat_freq_cm.groupby(['germany','match_rel','cl_cm_label']).freq.apply(np.sum).reset_index()
# should sum to one
cat_freq_cm_agg_cmpl.groupby(['germany']).freq.sum()